In [1]:
import importlib.util
import logging
import os
import random
import re
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

if importlib.util.find_spec("icu") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "libicu-dev", "build-essential"], check=True)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "duckdb",
        "pyarrow",
        "aiohttp",
        "fasttext",
        "pycld2",
        "morfessor",
        "PyICU",
        "polyglot",
    ],
    check=True,
)

import asyncio
import json

import aiohttp
import duckdb
import fasttext
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive, userdata
from polyglot.detect import Detector

drive.mount("/content/drive", force_remount=False)

DATA_DIR = Path("/content/drive/MyDrive/Language Detection")
PARQUET_PATH = DATA_DIR / "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"
TEMP_DIR = Path("/content/duckdb_tmp")

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(PARQUET_PATH)

TEMP_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET threads = {max(1, min(os.cpu_count() or 4, 8))}")
con.execute("SET memory_limit = '2GB'")
con.execute(f"SET temp_directory = '{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order = false")

logging.getLogger("polyglot.detect.base").setLevel(logging.ERROR)

parquet = pq.ParquetFile(PARQUET_PATH)
metadata = parquet.metadata

print("File :", PARQUET_PATH)
print("Size :", f"{PARQUET_PATH.stat().st_size / 1024**2:.2f} MB")
print("Rows :", f"{metadata.num_rows:,}")
print("Cols :", metadata.num_columns)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File : /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
Size : 469.49 MB
Rows : 3,469
Cols : 12


In [2]:
LANGUAGES = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt": "Portuguese",
    "es": "Spanish",
    "ru": "Russian",
}

MIN_WORDS = 4
MIN_SEGMENTS = 90
SESSIONS_PER_LANGUAGE = 5
SEGMENTS_PER_SESSION = 5
EXPECTED_BENCHMARK_ROWS = len(LANGUAGES) * SESSIONS_PER_LANGUAGE * SEGMENTS_PER_SESSION

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is missing from Colab Secrets")

MODELS = {
    "ox_alpha": {
        "id": "stealth/ox-alpha",
        "tier": "standard",
    },
    "nemotron_3_5_lightning": {
        "id": "nvidia/nemotron-3.5-lightning:free",
        "tier": "free",
    },
    "glm_5_2": {
        "id": "z-ai/glm-5.2:free",
        "tier": "free",
    },
    "gemma_4_31b": {
        "id": "google/gemma-4-31b-it:free",
        "tier": "free",
    },
    "hy_mt2_30b_a3b": {
        "id": "tencent/hy-mt2-30b-a3b",
        "tier": "standard",
    },
}

ALLOWED_LANGUAGES = set(LANGUAGES)
ALLOWED_OUTPUTS = ALLOWED_LANGUAGES | {"other"}

STANDARD_BATCH_SIZE = 75
FREE_BATCH_SIZE = 50
STANDARD_CONCURRENCY = 4
FREE_CONCURRENCY = 1
STANDARD_RETRIES = 1
FREE_RETRIES = 2
FREE_MIN_INTERVAL_SECONDS = 2.0
REQUEST_TIMEOUT_SECONDS = 60
MAX_OUTPUT_TOKENS = 1200

print("OpenRouter key: OK")
print("Benchmark rows expected:", EXPECTED_BENCHMARK_ROWS)


OpenRouter key: OK
Benchmark rows expected: 150


In [3]:
language_sql = ", ".join(f"'{code}'" for code in LANGUAGES)

con.execute(
    f"""
    CREATE OR REPLACE TABLE session_stats AS
    SELECT
        gamesession_id,
        user_id,
        game_name,
        url,
        model_type,
        TRY_CAST(created_at AS TIMESTAMP) AS created_at,
        lang_detected AS language_code,
        lang_probability,
        list_count(
            list_filter(
                transcript_segments,
                segment ->
                    segment.text IS NOT NULL
                    AND TRIM(segment.text) <> ''
                    AND segment.words IS NOT NULL
                    AND len(segment.words) >= {MIN_WORDS}
            )
        ) AS segment_count
    FROM read_parquet('{PARQUET_PATH.as_posix()}')
    WHERE
        lang_detected IN ({language_sql})
        AND transcript_segments IS NOT NULL
        AND len(transcript_segments) >= {MIN_SEGMENTS}
    """
)

con.execute(
    f"""
    CREATE OR REPLACE TABLE selected_sessions AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY language_code
                ORDER BY
                    segment_count ASC,
                    created_at DESC NULLS LAST,
                    gamesession_id DESC
            ) AS session_rank
        FROM session_stats
        WHERE segment_count >= {MIN_SEGMENTS}
    )
    SELECT
        language_code,
        session_rank,
        gamesession_id,
        user_id,
        game_name,
        url,
        model_type,
        created_at,
        lang_probability,
        segment_count
    FROM ranked
    WHERE session_rank <= {SESSIONS_PER_LANGUAGE}
    """
)

con.execute(
    f"""
    CREATE OR REPLACE TABLE selected_segments AS
    WITH source AS (
        SELECT
            p.gamesession_id,
            p.lang_detected AS language_code,
            p.transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}') AS p
        INNER JOIN selected_sessions AS s
            ON p.gamesession_id = s.gamesession_id
            AND p.lang_detected = s.language_code
    ),
    exploded AS (
        SELECT
            gamesession_id,
            language_code,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM source
    )
    SELECT
        gamesession_id,
        language_code,
        segment_index,
        TRIM(segment.text) AS segment_text,
        len(segment.words) AS word_count
    FROM exploded
    WHERE
        segment.text IS NOT NULL
        AND TRIM(segment.text) <> ''
        AND segment.words IS NOT NULL
        AND len(segment.words) >= {MIN_WORDS}
    """
)

selected_sessions = con.execute(
    """
    SELECT *
    FROM selected_sessions
    ORDER BY language_code, session_rank
    """
).df()

selected_sessions.insert(
    0,
    "language",
    selected_sessions["language_code"].map(LANGUAGES),
)

display(selected_sessions)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,language,language_code,session_rank,gamesession_id,user_id,game_name,url,model_type,created_at,lang_probability,segment_count
0,German,de,1,141268064,467269,Naraka,https://www.twitch.tv/videos/2854317619,gen10,2026-08-23 23:29:07,0.8291,98
1,German,de,2,141264210,399263,Call of Duty: Modern Warfare 4,https://www.twitch.tv/videos/2854157061,warfare4,2026-08-23 18:39:26,0.9072,107
2,German,de,3,141265651,687197,Escape from Tarkov,https://www.twitch.tv/videos/2854208944,gen10,2026-08-23 21:53:10,0.9199,114
3,German,de,4,141236998,833542,COD: Warzone3-2,https://www.twitch.tv/videos/2853489925,warfare2,2026-08-23 03:22:23,0.6826,115
4,German,de,5,141238626,812794,COD: Warzone3-2,https://www.twitch.tv/videos/2853620823,warfare2,2026-08-23 04:09:16,0.9678,118
5,English,en,1,141266766,395792,Phasmophobia,https://www.twitch.tv/videos/2854265768,gen10,2026-08-23 22:19:05,0.9463,90
6,English,en,2,141245486,773042,COD: Modern Warfare III,https://www.youtube.com/watch?v=UAUU-ciTr08,warfare3,2026-08-23 12:30:09,0.9277,90
7,English,en,3,141250470,615244,Helldivers 2,https://www.twitch.tv/videos/2853894951,gen4,2026-08-23 09:36:17,0.9976,90
8,English,en,4,141210406,466455,COD: Warzone3-2,https://www.twitch.tv/videos/2852833829,warfare2,2026-08-23 00:10:17,0.8442,90
9,English,en,5,141236416,100090,Call of Duty: Modern Warfare 4,https://www.twitch.tv/videos/2853520407,warfare4,2026-08-23 09:08:50,0.6890,91


In [4]:
session_validation = con.execute(
    f"""
    WITH session_checks AS (
        SELECT
            s.language_code,
            COUNT(*) AS session_count,
            MIN(s.segment_count) AS min_session_segments,
            COUNT(*) = {SESSIONS_PER_LANGUAGE} AS has_required_sessions,
            MIN(s.segment_count) >= {MIN_SEGMENTS} AS sessions_meet_minimum
        FROM selected_sessions AS s
        GROUP BY s.language_code
    ),
    segment_checks AS (
        SELECT
            language_code,
            COUNT(DISTINCT gamesession_id) AS validated_sessions,
            MIN(word_count) AS min_segment_words,
            MIN(word_count) >= {MIN_WORDS} AS segments_meet_minimum
        FROM selected_segments
        GROUP BY language_code
    ),
    order_checks AS (
        SELECT
            language_code,
            bool_and(
                next_segment_count IS NULL
                OR segment_count <= next_segment_count
            ) AS smallest_session_first
        FROM (
            SELECT
                language_code,
                session_rank,
                segment_count,
                lead(segment_count) OVER (
                    PARTITION BY language_code
                    ORDER BY session_rank
                ) AS next_segment_count
            FROM selected_sessions
        )
        GROUP BY language_code
    )
    SELECT
        s.language_code,
        s.session_count,
        s.min_session_segments,
        g.min_segment_words,
        s.has_required_sessions,
        s.sessions_meet_minimum,
        g.segments_meet_minimum,
        o.smallest_session_first,
        (
            s.has_required_sessions
            AND s.sessions_meet_minimum
            AND g.segments_meet_minimum
            AND o.smallest_session_first
        ) AS all_checks_passed
    FROM session_checks AS s
    INNER JOIN segment_checks AS g USING (language_code)
    INNER JOIN order_checks AS o USING (language_code)
    ORDER BY language_code
    """
).df()

session_validation.insert(
    0,
    "language",
    session_validation["language_code"].map(LANGUAGES),
)

display(session_validation)

if len(session_validation) != len(LANGUAGES):
    raise ValueError("Validation did not cover all target languages")

if not session_validation["all_checks_passed"].all():
    raise ValueError("Session validation failed")

print("Session validation passed.")


,language,language_code,session_count,min_session_segments,min_segment_words,has_required_sessions,sessions_meet_minimum,segments_meet_minimum,smallest_session_first,all_checks_passed
0,German,de,5,98,4,True,True,True,True,True
1,English,en,5,90,4,True,True,True,True,True
2,Spanish,es,5,94,4,True,True,True,True,True
3,French,fr,5,130,4,True,True,True,True,True
4,Portuguese,pt,5,91,4,True,True,True,True,True
5,Russian,ru,5,90,4,True,True,True,True,True


Session validation passed.


In [5]:
benchmark_segments = con.execute(
    f"""
    WITH ranked AS (
        SELECT
            s.language_code,
            s.gamesession_id,
            s.session_rank,
            g.segment_index,
            g.segment_text,
            g.word_count,
            row_number() OVER (
                PARTITION BY g.gamesession_id
                ORDER BY g.segment_index
            ) AS segment_position,
            count(*) OVER (
                PARTITION BY g.gamesession_id
            ) AS session_segment_count
        FROM selected_segments AS g
        INNER JOIN selected_sessions AS s
            ON g.gamesession_id = s.gamesession_id
            AND g.language_code = s.language_code
    ),
    targets AS (
        SELECT
            *,
            round(
                segment_position
                * ({SEGMENTS_PER_SESSION} + 1.0)
                / (session_segment_count + 1.0)
            ) AS sample_bucket
        FROM ranked
    ),
    sampled AS (
        SELECT *
        FROM targets
        WHERE sample_bucket BETWEEN 1 AND {SEGMENTS_PER_SESSION}
        QUALIFY row_number() OVER (
            PARTITION BY gamesession_id, sample_bucket
            ORDER BY
                abs(
                    segment_position
                    - sample_bucket
                    * (session_segment_count + 1.0)
                    / ({SEGMENTS_PER_SESSION} + 1.0)
                ),
                segment_index
        ) = 1
    )
    SELECT
        language_code AS dataset_language,
        gamesession_id,
        segment_index,
        segment_text,
        word_count
    FROM sampled
    ORDER BY
        dataset_language,
        session_rank,
        segment_index
    """
).df()

sample_counts = (
    benchmark_segments
    .groupby(["dataset_language", "gamesession_id"])
    .size()
)

if len(benchmark_segments) != EXPECTED_BENCHMARK_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_BENCHMARK_ROWS} benchmark rows, found {len(benchmark_segments)}"
    )

if not (sample_counts == SEGMENTS_PER_SESSION).all():
    raise ValueError("Every selected session must contribute exactly 5 benchmark segments")

if benchmark_segments["word_count"].min() < MIN_WORDS:
    raise ValueError("Benchmark contains a segment below the minimum word count")

display(benchmark_segments)


,dataset_language,gamesession_id,segment_index,segment_text,word_count
0,de,141268064,25,Alter. Einfach gleiche Stun.,4
1,de,141268064,46,"Ja irgendwie schon, Alter, aber es macht weh.",8
2,de,141268064,63,"Ah, ja. Ja, voll. Ja, das sicher ein spannende...",10
3,de,141268064,87,"Legende eins, let´s go! Das guckt nicht mehr.",8
4,de,141268064,106,"Bro, ich hasse es. War das der Heng? Heng und ...",13
...,...,...,...,...,...
145,ru,141262832,43,Поэтому я... в возврат кинул хуйню.,6
146,ru,141262832,73,еще далинка нужна очень сильно мне,6
147,ru,141262832,103,Хочу на Леончике карточку сыграть. Родировку. ...,11
148,ru,141262832,132,"вот меня палец есть, у него палец раскачу",8


In [6]:
FASTTEXT_MODEL_PATH = Path("/content/lid.176.ftz")

if not FASTTEXT_MODEL_PATH.exists():
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz",
        FASTTEXT_MODEL_PATH,
    )

FASTTEXT_MODEL = fasttext.load_model(FASTTEXT_MODEL_PATH.as_posix())


def normalize_language(value):
    value = str(value).strip().lower()
    value = value.replace("__label__", "")
    value = value.split("-")[0].split("_")[0]
    return value if value in ALLOWED_OUTPUTS else "other"


def normalize_text(text):
    if pd.isna(text):
        return ""
    return " ".join(str(text).replace("\n", " ").replace("\r", " ").split())


def predict_fasttext(text):
    text = normalize_text(text)
    if not text:
        return "other"
    predictions = FASTTEXT_MODEL.f.predict(text, 1, 0.0, "strict")
    if not predictions:
        return "other"
    _, label = predictions[0]
    return normalize_language(label)


def predict_polyglot(text):
    text = normalize_text(text)
    if not text:
        return "other"
    try:
        return normalize_language(Detector(text, quiet=True).language.code)
    except Exception:
        return "other"


benchmark_results = benchmark_segments.copy()
benchmark_results["fasttext_language"] = benchmark_results["segment_text"].map(predict_fasttext)
benchmark_results["polyglot_language"] = benchmark_results["segment_text"].map(predict_polyglot)

display(benchmark_results)


,dataset_language,gamesession_id,segment_index,segment_text,word_count,fasttext_language,polyglot_language
0,de,141268064,25,Alter. Einfach gleiche Stun.,4,de,de
1,de,141268064,46,"Ja irgendwie schon, Alter, aber es macht weh.",8,de,de
2,de,141268064,63,"Ah, ja. Ja, voll. Ja, das sicher ein spannende...",10,de,de
3,de,141268064,87,"Legende eins, let´s go! Das guckt nicht mehr.",8,de,de
4,de,141268064,106,"Bro, ich hasse es. War das der Heng? Heng und ...",13,de,de
...,...,...,...,...,...,...,...
145,ru,141262832,43,Поэтому я... в возврат кинул хуйню.,6,ru,ru
146,ru,141262832,73,еще далинка нужна очень сильно мне,6,ru,ru
147,ru,141262832,103,Хочу на Леончике карточку сыграть. Родировку. ...,11,ru,ru
148,ru,141262832,132,"вот меня палец есть, у него палец раскачу",8,ru,other


In [9]:
SYSTEM_PROMPT = """
You are a language identification classifier.

For every transcript, identify the primary language represented by the text.

Allowed outputs only:
en
de
fr
pt
es
ru
other

Return exactly one prediction for every input id.
Preferred format:
{"predictions":[{"id":0,"language":"en"}]}

You may alternatively return one line per item as:
0|en
1|de

Do not translate.
Do not explain.
Do not correct the transcript.
Do not infer from any external context.
""".strip()


class AsyncRateLimiter:
    def __init__(self, min_interval_seconds):
        self.min_interval_seconds = min_interval_seconds
        self.lock = asyncio.Lock()
        self.last_request_at = 0.0

    async def wait(self):
        if self.min_interval_seconds <= 0:
            return
        async with self.lock:
            now = asyncio.get_running_loop().time()
            delay = self.min_interval_seconds - (now - self.last_request_at)
            if delay > 0:
                await asyncio.sleep(delay)
            self.last_request_at = asyncio.get_running_loop().time()


free_rate_limiter = AsyncRateLimiter(FREE_MIN_INTERVAL_SECONDS)
standard_semaphore = asyncio.Semaphore(STANDARD_CONCURRENCY)
free_semaphore = asyncio.Semaphore(FREE_CONCURRENCY)


def build_records(frame):
    work = frame.reset_index(drop=True).copy()
    work["row_id"] = work.index.astype(int)
    return work


def split_batches(records, size):
    return [records[i:i + size] for i in range(0, len(records), size)]


def build_prompt(batch):
    payload = [
        {
            "id": int(row["row_id"]),
            "text": row["segment_text"],
        }
        for row in batch
    ]
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))


def extract_message_text(message):
    content = message.get("content", "")
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                text = item.get("text") or item.get("content") or ""
                if text:
                    parts.append(str(text))
        return "\n".join(parts).strip()
    if content is None:
        return ""
    return str(content).strip()


def extract_json_candidates(content):
    text = str(content).strip()
    candidates = [text]
    fenced = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE)
    if fenced != text:
        candidates.append(fenced)
    first_object = fenced.find("{")
    last_object = fenced.rfind("}")
    if first_object >= 0 and last_object > first_object:
        candidates.append(fenced[first_object:last_object + 1])
    first_array = fenced.find("[")
    last_array = fenced.rfind("]")
    if first_array >= 0 and last_array > first_array:
        candidates.append(fenced[first_array:last_array + 1])
    return list(dict.fromkeys(candidates))


def parse_json_predictions(payload, expected_ids):
    results = {}
    positional = []

    def consume(value):
        if isinstance(value, dict):
            container_keys = ("predictions", "results", "items", "data", "output")
            for key in container_keys:
                if key in value:
                    consume(value[key])
            id_value = value.get("id", value.get("row_id", value.get("index")))
            language_value = value.get(
                "language",
                value.get("lang", value.get("code", value.get("prediction"))),
            )
            if id_value is not None and language_value is not None:
                try:
                    row_id = int(id_value)
                    language = normalize_language(language_value)
                    if row_id in expected_ids:
                        results[row_id] = language
                except Exception:
                    pass
            if not any(key in value for key in container_keys):
                numeric_mapping = True
                mapped = {}
                for key, item in value.items():
                    try:
                        row_id = int(key)
                    except Exception:
                        numeric_mapping = False
                        break
                    if isinstance(item, str):
                        mapped[row_id] = normalize_language(item)
                    else:
                        numeric_mapping = False
                        break
                if numeric_mapping:
                    for row_id, language in mapped.items():
                        if row_id in expected_ids:
                            results[row_id] = language
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, str):
                    positional.append(normalize_language(item))
                else:
                    consume(item)
        elif isinstance(value, str):
            positional.append(normalize_language(value))

    consume(payload)
    return results, positional


def parse_predictions(content, expected_ids):
    expected_ids = list(expected_ids)
    expected_set = set(expected_ids)

    for candidate in extract_json_candidates(content):
        try:
            payload = json.loads(candidate)
        except Exception:
            continue
        results, positional = parse_json_predictions(payload, expected_set)
        if results:
            return results, "json_id"
        if len(positional) == len(expected_ids):
            return dict(zip(expected_ids, positional)), "json_positional"

    patterns = [
        r"(?mi)^\s*(\d+)\s*(?:\||\t|:|;|,|->|=>|-)\s*[\"']?(en|de|fr|pt|es|ru|other)[\"']?\s*[.,;]?$",
        r"(?mi)^\s*(\d+)\s*[.)]\s*[\"']?(en|de|fr|pt|es|ru|other)[\"']?\s*[.,;]?$",
        r"(?i)[\"']?(?:id|row_id|index)[\"']?\s*[:=]\s*(\d+).*?[\"']?(?:language|lang|code|prediction)[\"']?\s*[:=]\s*[\"']?(en|de|fr|pt|es|ru|other)[\"']?",
    ]

    results = {}
    for pattern in patterns:
        for row_id, language in re.findall(pattern, str(content)):
            row_id = int(row_id)
            if row_id in expected_set:
                results[row_id] = normalize_language(language)
    if results:
        return results, "regex_id"

    language_only_lines = []
    for line in str(content).splitlines():
        match = re.fullmatch(
            r"\s*(?:[-*]\s*)?[\"']?(en|de|fr|pt|es|ru|other)[\"']?\s*[.,;]?\s*",
            line,
            flags=re.IGNORECASE,
        )
        if match:
            language_only_lines.append(normalize_language(match.group(1)))

    if len(language_only_lines) == len(expected_ids):
        return dict(zip(expected_ids, language_only_lines)), "line_positional"

    if len(expected_ids) == 1:
        tokens = re.findall(r"\b(en|de|fr|pt|es|ru|other)\b", str(content), flags=re.IGNORECASE)
        if len(tokens) == 1:
            return {expected_ids[0]: normalize_language(tokens[0])}, "single_language"

    return {}, "unparsed"


async def request_openrouter(session, model_name, model_id, tier, batch):
    semaphore = free_semaphore if tier == "free" else standard_semaphore
    retries = FREE_RETRIES if tier == "free" else STANDARD_RETRIES

    payload = {
        "model": model_id,
        "temperature": 0,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(batch)},
        ],
    }

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    last_error = None

    for attempt in range(retries + 1):
        if tier == "free":
            await free_rate_limiter.wait()

        try:
            async with semaphore:
                async with session.post(
                    OPENROUTER_URL,
                    headers=headers,
                    json=payload,
                ) as response:
                    raw_text = await response.text()
                    try:
                        data = json.loads(raw_text)
                    except Exception:
                        data = {}

                    if response.status == 200:
                        message = data.get("choices", [{}])[0].get("message", {})
                        return {
                            "http_status": response.status,
                            "message": message,
                            "content": extract_message_text(message),
                            "error": None,
                            "attempt": attempt + 1,
                        }

                    message = data.get("error", {}).get("message", raw_text[:500])
                    last_error = f"HTTP {response.status}: {message}"

                    if response.status not in {429, 500, 502, 503, 504}:
                        break

                    retry_after = response.headers.get("Retry-After")
                    if retry_after:
                        try:
                            delay = float(retry_after)
                        except Exception:
                            delay = 0.0
                    else:
                        base = 10.0 if tier == "free" else 2.0
                        delay = min(60.0, base * (2 ** attempt) + random.random())

        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            base = 10.0 if tier == "free" else 2.0
            delay = min(60.0, base * (2 ** attempt) + random.random())

        if attempt < retries:
            await asyncio.sleep(delay)

    return {
        "http_status": None,
        "message": {},
        "content": "",
        "error": last_error or "Unknown request failure",
        "attempt": retries + 1,
    }


async def classify_batch(
    session,
    model_name,
    model_config,
    batch,
):
    expected_ids = [
        int(row["row_id"])
        for row in batch
    ]

    response = await request_openrouter(
        session,
        model_name,
        model_config["id"],
        model_config["tier"],
        batch,
    )

    content = response["content"]

    if content:
        results, parser_mode = parse_predictions(
            content,
            expected_ids,
        )
    else:
        results = {}
        parser_mode = "no_content"

    missing_ids = [
        row_id
        for row_id in expected_ids
        if row_id not in results
    ]

    log_entry = {
        "model_name": model_name,
        "model_id": model_config["id"],
        "tier": model_config["tier"],
        "batch_size": len(batch),
        "http_status": response["http_status"],
        "attempts": response["attempt"],
        "parser_mode": parser_mode,
        "parsed_count": len(results),
        "expected_count": len(expected_ids),
        "missing_count": len(missing_ids),
        "error": response["error"],
        "raw_content_preview": content[:4000],
        "raw_message_preview": json.dumps(
            response["message"],
            ensure_ascii=False,
        )[:4000],
    }

    return model_name, results, log_entry


async def run_model_benchmark(frame):
    work = build_records(frame)

    records = work[
        ["row_id", "segment_text"]
    ].to_dict("records")

    prediction_store = {
        model_name: {}
        for model_name in MODELS
    }

    request_logs = []

    timeout = aiohttp.ClientTimeout(
        total=REQUEST_TIMEOUT_SECONDS
    )

    connector = aiohttp.TCPConnector(
        limit=STANDARD_CONCURRENCY + FREE_CONCURRENCY,
        ttl_dns_cache=300,
    )

    async with aiohttp.ClientSession(
        timeout=timeout,
        connector=connector,
    ) as session:
        tasks = []

        for model_name, model_config in MODELS.items():
            batch_size = (
                FREE_BATCH_SIZE
                if model_config["tier"] == "free"
                else STANDARD_BATCH_SIZE
            )

            batches = split_batches(
                records,
                batch_size,
            )

            for batch in batches:
                tasks.append(
                    asyncio.create_task(
                        classify_batch(
                            session,
                            model_name,
                            model_config,
                            batch,
                        )
                    )
                )

        total = len(tasks)

        for index, task in enumerate(
            asyncio.as_completed(tasks),
            start=1,
        ):
            model_name, results, log_entry = await task

            prediction_store[
                model_name
            ].update(results)

            request_logs.append(
                log_entry
            )

            print(
                f"\rCompleted {index}/{total}",
                end="",
            )

    print()

    output = work.drop(
        columns="row_id"
    ).copy()

    for model_name in MODELS:
        output[
            f"{model_name}_language"
        ] = [
            prediction_store[
                model_name
            ].get(
                row_id,
                "error",
            )
            for row_id in range(len(output))
        ]

    request_log = pd.DataFrame(
        request_logs
    )

    return output, request_log


In [10]:
benchmark_results, model_request_log = await run_model_benchmark(benchmark_results)

model_status = []

for model_name in MODELS:
    column = f"{model_name}_language"
    successful = benchmark_results[column].ne("error").sum()
    failed = benchmark_results[column].eq("error").sum()
    agreement = (
        benchmark_results.loc[benchmark_results[column].ne("error"), column]
        == benchmark_results.loc[benchmark_results[column].ne("error"), "dataset_language"]
    ).mean()

    model_status.append(
        {
            "model_name": model_name,
            "successful_predictions": int(successful),
            "failed_predictions": int(failed),
            "dataset_agreement_rate": None if pd.isna(agreement) else round(float(agreement), 4),
        }
    )

model_status = pd.DataFrame(model_status)

display(model_status)

diagnostic_failures = model_request_log[
    (model_request_log["error"].notna())
    | (model_request_log["missing_count"] > 0)
].reset_index(drop=True)

display(
    diagnostic_failures[
        [
            "model_name",
            "model_id",
            "tier",
            "batch_size",
            "depth",
            "http_status",
            "attempts",
            "parser_mode",
            "parsed_count",
            "expected_count",
            "missing_count",
            "error",
            "raw_content_preview",
            "raw_message_preview",
        ]
    ]
)


Completed 25/25


,model_name,successful_predictions,failed_predictions,dataset_agreement_rate
0,ox_alpha,2,148,1.0000
1,nemotron_3_5_lightning,0,150,NaN
2,glm_5_2,0,150,NaN
3,gemma_4_31b,0,150,NaN
4,hy_mt2_30b_a3b,150,0,0.8933


KeyError: "['depth'] not in index"

In [11]:
MODEL_COLUMNS = [f"{name}_language" for name in MODELS]

mismatch_audit = benchmark_results[
    MODEL_COLUMNS
].ne(
    benchmark_results["dataset_language"],
    axis=0,
)

mismatch_rows = benchmark_results[
    mismatch_audit.any(axis=1)
].copy()

russian_audit = benchmark_results[
    benchmark_results["dataset_language"].eq("ru")
].copy()

print("Rows with at least one OpenRouter disagreement:", len(mismatch_rows))
display(mismatch_rows)

print("Russian benchmark rows:")
display(russian_audit)

hy_mt2_request_evidence = model_request_log[
    model_request_log["model_name"].eq("hy_mt2_30b_a3b")
][
    [
        "model_name",
        "batch_size",
        "http_status",
        "parser_mode",
        "parsed_count",
        "missing_count",
        "raw_content_preview",
    ]
].reset_index(drop=True)

display(hy_mt2_request_evidence)


Rows with at least one OpenRouter disagreement: 150


,dataset_language,gamesession_id,segment_index,segment_text,word_count,fasttext_language,polyglot_language,ox_alpha_language,nemotron_3_5_lightning_language,glm_5_2_language,gemma_4_31b_language,hy_mt2_30b_a3b_language
0,de,141268064,25,Alter. Einfach gleiche Stun.,4,de,de,de,error,error,error,de
1,de,141268064,46,"Ja irgendwie schon, Alter, aber es macht weh.",8,de,de,de,error,error,error,de
2,de,141268064,63,"Ah, ja. Ja, voll. Ja, das sicher ein spannende...",10,de,de,error,error,error,error,de
3,de,141268064,87,"Legende eins, let´s go! Das guckt nicht mehr.",8,de,de,error,error,error,error,de
4,de,141268064,106,"Bro, ich hasse es. War das der Heng? Heng und ...",13,de,de,error,error,error,error,de
...,...,...,...,...,...,...,...,...,...,...,...,...
145,ru,141262832,43,Поэтому я... в возврат кинул хуйню.,6,ru,ru,error,error,error,error,ru
146,ru,141262832,73,еще далинка нужна очень сильно мне,6,ru,ru,error,error,error,error,ru
147,ru,141262832,103,Хочу на Леончике карточку сыграть. Родировку. ...,11,ru,ru,error,error,error,error,ru
148,ru,141262832,132,"вот меня палец есть, у него палец раскачу",8,ru,other,error,error,error,error,ru


Russian benchmark rows:


,dataset_language,gamesession_id,segment_index,segment_text,word_count,fasttext_language,polyglot_language,ox_alpha_language,nemotron_3_5_lightning_language,glm_5_2_language,gemma_4_31b_language,hy_mt2_30b_a3b_language
125,ru,141260682,17,"тебя C4, Парк! Убuros 33-й парк!",7,ru,ru,error,error,error,error,ru
126,ru,141260682,39,"Встретили! Встретили! Есть кнопка на контроле,...",8,ru,ru,error,error,error,error,ru
127,ru,141260682,55,"Я с тобой! Дверь к реакторе 2, первый на вашей...",14,ru,other,error,error,error,error,ru
128,ru,141260682,74,Пошли вверх. Расскажите координаты.,4,ru,ru,error,error,error,error,ru
129,ru,141260682,100,ты не принадлежишь на боевом,5,ru,ru,error,error,error,error,ru
130,ru,141228447,18,ну это было близко,4,ru,ru,error,error,error,error,ru
131,ru,141228447,38,Да. Узкие дырочки в деле!,5,ru,ru,error,error,error,error,ru
132,ru,141228447,57,ОГО да дай бускую дурачку проехал уй,7,ru,ru,error,error,error,error,ru
133,ru,141228447,76,Это дерьмо какое-то.,4,ru,ru,error,error,error,error,ru
134,ru,141228447,101,"Ну, сразу оставай меньше. Ай, блять, чего?",7,ru,ru,error,error,error,error,ru


,model_name,batch_size,http_status,parser_mode,parsed_count,missing_count,raw_content_preview
0,hy_mt2_30b_a3b,30,200.0,json_id,30,0,"{""predictions"":[{""id"":30,""language"":""en""},{""id..."
1,hy_mt2_30b_a3b,30,200.0,json_id,30,0,"{""predictions"":[{""id"":0,""language"":""de""},{""id""..."
2,hy_mt2_30b_a3b,30,200.0,json_id,30,0,"{""predictions"":[{""id"":60,""language"":""es""},{""id..."
3,hy_mt2_30b_a3b,30,200.0,json_id,30,0,"{""predictions"":[{""id"":90,""language"":""fr""},{""id..."
4,hy_mt2_30b_a3b,30,200.0,json_id,30,0,"{""predictions"":[{""id"":120,""language"":""pt""},{""i..."


In [12]:
final_benchmark = benchmark_results[
    [
        "dataset_language",
        "gamesession_id",
        "segment_index",
        "segment_text",
        "ox_alpha_language",
        "nemotron_3_5_lightning_language",
        "glm_5_2_language",
        "gemma_4_31b_language",
        "hy_mt2_30b_a3b_language",
        "fasttext_language",
        "polyglot_language",
    ]
].copy()

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(final_benchmark)


,dataset_language,gamesession_id,segment_index,segment_text,ox_alpha_language,nemotron_3_5_lightning_language,glm_5_2_language,gemma_4_31b_language,hy_mt2_30b_a3b_language,fasttext_language,polyglot_language
0,de,141268064,25,Alter. Einfach gleiche Stun.,de,error,error,error,de,de,de
1,de,141268064,46,"Ja irgendwie schon, Alter, aber es macht weh.",de,error,error,error,de,de,de
2,de,141268064,63,"Ah, ja. Ja, voll. Ja, das sicher ein spannendes Battle.",error,error,error,error,de,de,de
3,de,141268064,87,"Legende eins, let´s go! Das guckt nicht mehr.",error,error,error,error,de,de,de
4,de,141268064,106,"Bro, ich hasse es. War das der Heng? Heng und Gauntlet, glaube ich.",error,error,error,error,de,de,de
5,de,141264210,25,"Es ist ja schon langweilig, weil ich keine Videos gegen mich mehr machen",error,error,error,error,de,de,de
6,de,141264210,44,killergen was geht was geht,error,error,error,error,de,en,en
7,de,141264210,64,"Ja, Bruder, Simon, bist du im Discord?",error,error,error,error,de,de,de
8,de,141264210,88,"hab Ist aber schnell nicht zu machen. Äh, stimmt, stimmt, stimmt, stimmt.",error,error,error,error,de,de,de
9,de,141264210,107,"Indem, dass du Control und C trittst, eh V.",error,error,error,error,de,de,de


In [13]:
OUTPUT_DIR = DATA_DIR / "benchmark_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = OUTPUT_DIR / "language_detection_model_benchmark.csv"
RESULTS_PARQUET = OUTPUT_DIR / "language_detection_model_benchmark.parquet"
REQUEST_LOG_CSV = OUTPUT_DIR / "language_detection_model_request_log.csv"
MODEL_STATUS_CSV = OUTPUT_DIR / "language_detection_model_status.csv"

final_benchmark.to_csv(RESULTS_CSV, index=False)
final_benchmark.to_parquet(RESULTS_PARQUET, index=False)
model_request_log.to_csv(REQUEST_LOG_CSV, index=False)
model_status.to_csv(MODEL_STATUS_CSV, index=False)

print("Results CSV   :", RESULTS_CSV)
print("Results Parquet:", RESULTS_PARQUET)
print("Request log   :", REQUEST_LOG_CSV)
print("Model status  :", MODEL_STATUS_CSV)


Results CSV   : /content/drive/MyDrive/Language Detection/benchmark_outputs/language_detection_model_benchmark.csv
Results Parquet: /content/drive/MyDrive/Language Detection/benchmark_outputs/language_detection_model_benchmark.parquet
Request log   : /content/drive/MyDrive/Language Detection/benchmark_outputs/language_detection_model_request_log.csv
Model status  : /content/drive/MyDrive/Language Detection/benchmark_outputs/language_detection_model_status.csv
